# Who wants to be a PoliMillionaire? — Multi-Agent System

> **Assistenti di codice usati:** Claude Sonnet 4.6 (Anthropic)

Sistema multi-agent con **Orchestrator → MathAgent | KnowledgeAgent**.
Logging completo compatibile con le celle di analisi di `analysis.ipynb`.

## 1 · Environment & Imports

In [ ]:
import json
import os
import re
import sys
import time
import random
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Rilevamento ambiente ──────────────────────────────────────────────────
IN_COLAB  = False
IN_KAGGLE = False

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/working"):
    IN_KAGGLE = True
else:
    try:
        import google.colab
        if os.path.exists("/var/colab/hostname"):
            IN_COLAB = True
    except ImportError:
        pass

# ── Percorsi ─────────────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    print("Monto Google Drive...")
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/NLP/NLP_project")
elif IN_KAGGLE:
    BASE_DIR = Path("/kaggle/working/NLP_project")
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("./")

if BASE_DIR.exists():
    os.chdir(BASE_DIR)
    print(f"Cartella di lavoro: {os.getcwd()}")
else:
    print(f"Percorso non trovato: {BASE_DIR}")

if str(BASE_DIR.resolve()) not in sys.path:
    sys.path.append(str(BASE_DIR.resolve()))

RUNS_DIR = Path("./data/runs")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Environment:", "KAGGLE" if IN_KAGGLE else "COLAB" if IN_COLAB else "LOCAL")
print("Runs directory:", RUNS_DIR)

## 2 · Installazione dipendenze

In [ ]:
!pip install -q colab-xterm
!pip install -q \
    langchain langchain-community langchain-ollama langgraph \
    wikipedia ddgs python-dotenv

%load_ext colabxterm

## 3 · Terminale (avvia Ollama)

Esegui nel terminale aperto da `%xterm`:
```bash
sudo apt-get install -y zstd
curl -fsSL https://ollama.com/install.sh | sh
ollama serve & ollama pull qwen2.5:7b
```

In [ ]:
%xterm

## 4 · Configurazione

Modifica `MODEL_CONFIG` per assegnare modelli Ollama diversi ai singoli agenti.
`GAME_USER` e `GAME_PASS` vengono letti da variabili d'ambiente (file `.env` o export).

In [ ]:
import dotenv
dotenv.load_dotenv()

# ── Modello per agente ────────────────────────────────────────────────────
# Cambia il modello di un singolo agente senza toccare gli altri.
MODEL_CONFIG = {
    "orchestrator": "qwen2.5:7b",
    "math":         "qwen2.5:7b",
    "knowledge":    "qwen2.5:7b",
}

# ── Parametri del run ─────────────────────────────────────────────────────
REASONING   = False                               # True per il thinking mode
NUM_RUNS    = 3                                   # partite per competizione
MAX_LEVELS  = 15                                  # livelli massimi (fisso)
RUN_COMMENT = "multi-agent: orchestrator + math + knowledge"
TECHNIQUES  = ["multi-agent", "RAG-wikipedia", "RAG-duckduckgo", "tool-calling"]

# ── Credenziali API ───────────────────────────────────────────────────────
# GAME_USER / GAME_PASS evitano la collisione con $USER di sistema (Linux/Colab).
API_URL   = "http://131.175.15.22:51111/"
GAME_USER = os.getenv("USER")
GAME_PASS = os.getenv("PASSWORD")

print("MODEL_CONFIG:", MODEL_CONFIG)
print(f"NUM_RUNS: {NUM_RUNS}  |  MAX_LEVELS: {MAX_LEVELS}")
print("GAME_USER:", GAME_USER)

## 5 · Funzioni di logging

Identiche a `analysis.ipynb` (compatibilità completa con le celle di analisi).

**Modifiche rispetto all'originale:**
- `save_metadata` accetta `model_config` (dict) invece di `model_name` (str)
- `update_runs_log` include il campo `model_config` nell'entry del log

In [ ]:
def _sanitize(text, max_len=24):
    # Rende la stringa sicura per il filesystem
    return "".join(c if c.isalnum() or c in "_-" else "_" for c in text)[:max_len].strip("_")


def create_run_folder(model_name):
    ts = datetime.now().strftime("%d%m%Y%H%M")
    name = _sanitize(model_name) + "_" + ts
    run_dir = RUNS_DIR / name
    run_dir.mkdir(parents=True, exist_ok=True)
    print("Run folder created:", run_dir.name)
    return run_dir


def save_metadata(run_dir, model_config, reasoning, comment, techniques, num_runs):
    """Scrive .run_info.json subito — sopravvive a un crash a metà run."""
    meta = {
        "model_name":               model_config.get("orchestrator", "unknown"),
        "model_config":             model_config,
        "reasoning":                reasoning,
        "comment":                  comment,
        "techniques":               techniques,
        "timestamp":                datetime.now().isoformat(),
        "num_runs_per_competition": num_runs,
    }
    with open(run_dir / ".run_info.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    return meta


def compute_competition_summary(attempts_log):
    """Accuracy = correct / (num_attempts × MAX_LEVELS). Penalizza uscite precoci."""
    total_q = total_l = correct_q = timeout_q = 0
    total_time = 0.0
    all_earned = []
    max_level_reached = 0
    for attempt in attempts_log:
        if attempt["final_level"] > max_level_reached:
            max_level_reached = attempt["final_level"]
        total_l += attempt["final_level"]
        all_earned.append(attempt["earned"])
        for q in attempt["questions"]:
            total_q += 1
            total_time += q["time"]
            if q["timed_out"]:
                timeout_q += 1
            elif q["correct"] is True:
                correct_q += 1
    denominator = len(attempts_log) * MAX_LEVELS
    return {
        "total_attempts":        len(attempts_log),
        "questions_answered":    total_q,
        "avg_level":             round(total_l / len(attempts_log), 2) if attempts_log else 0,
        "max_level":             max_level_reached,
        "avg_earned":            round(sum(all_earned) / len(all_earned), 2) if all_earned else 0,
        "max_earned":            max(all_earned) if all_earned else 0,
        "accuracy":              round(correct_q / denominator, 4) if denominator > 0 else 0,
        "timeout_count":         timeout_q,
        "avg_time_per_question": round(total_time / total_q, 4) if total_q > 0 else 0,
        "max_time":              round(max(q["time"] for a in attempts_log for q in a["questions"]), 4) if total_q > 0 else 0,
    }


def save_competition_result(run_dir, comp, attempts_log):
    filename = "comp_" + str(comp.id) + "_" + _sanitize(comp.name) + ".json"
    data = {
        "competition_id":   comp.id,
        "competition_name": comp.name,
        "summary":          compute_competition_summary(attempts_log),
        "attempts":         attempts_log,
    }
    with open(run_dir / filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print("  Saved:", filename)


def save_run_summary(run_dir, all_comp_data, metadata):
    correct_total = denominator_total = 0
    per_comp = {}
    for comp_id, comp_info in all_comp_data.items():
        s = compute_competition_summary(comp_info["attempts"])
        per_comp[str(comp_id)] = {
            "name":      comp_info["name"],
            "accuracy":  s["accuracy"],
            "avg_level": s["avg_level"],
            "avg_earned": s["avg_earned"],
        }
        denominator_total += s["total_attempts"] * comp_info["max_levels"]
        for attempt in comp_info["attempts"]:
            for q in attempt["questions"]:
                if q["correct"] is True:
                    correct_total += 1
    run_info = {
        **metadata,
        "total_competitions": len(all_comp_data),
        "overall_accuracy":   round(correct_total / denominator_total, 4) if denominator_total > 0 else 0,
        "comp_info":          per_comp,
    }
    with open(run_dir / ".run_info.json", "w", encoding="utf-8") as f:
        json.dump(run_info, f, indent=2)
    print("  Saved: .run_info.json")
    return run_info


def update_runs_log(runs_dir, run_folder, run_info):
    log_path = runs_dir / "runs_log.json"
    entries = []
    if log_path.exists():
        with open(log_path, "r", encoding="utf-8") as f:
            entries = json.load(f)
    competition_summary = {
        info["name"]: info["accuracy"]
        for info in run_info["comp_info"].values()
    }
    entries.append({
        "run_folder":           run_folder,
        "model_name":           run_info["model_name"],
        "model_config":         run_info.get("model_config", {}),
        "total_runs":           run_info["num_runs_per_competition"],
        "overall_accuracy":     run_info["overall_accuracy"],
        "reasoning":            run_info["reasoning"],
        "comment":              run_info["comment"],
        "techniques":           run_info["techniques"],
        "competition_accuracy": competition_summary,
    })
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(entries, f, indent=2)
    print("runs_log.json updated —", len(entries), "runs logged")

print("Logging functions loaded")

## 6 · Client API

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

client = MillionaireClient(API_URL)
try:
    user = client.login(GAME_USER, GAME_PASS)
    print("Login successful. Welcome,", user.username)
except AuthenticationError as e:
    print("Login failed:", e)

## 7 · Utilità comuni

`make_model` è il punto centrale per istanziare i modelli.
`_extract_agent_trace` e `print_agent_stats` gestiscono debug e traccia.

In [ ]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.tools import render_text_description


def make_model(model_name: str, temperature: float = 0.0) -> ChatOllama:
    """Istanzia un ChatOllama. Punto centrale per modificare i parametri del modello."""
    return ChatOllama(model=model_name, temperature=temperature)


def _extract_agent_trace(result: dict, model_name: str) -> dict:
    """
    Estrae la traccia di esecuzione da un risultato di agente LangGraph.

    Returns
    -------
    dict con campi:
        model        — nome del modello usato
        raw_output   — testo grezzo dell'ultimo AIMessage
        tools_called — lista di {name, args} per ogni tool invocato
    """
    messages = result.get("messages", [])
    raw = messages[-1].content if messages else ""
    tools_called = [
        {"name": tc["name"], "args": tc["args"]}
        for msg in messages
        if hasattr(msg, "tool_calls") and msg.tool_calls
        for tc in msg.tool_calls
    ]
    return {"model": model_name, "raw_output": raw, "tools_called": tools_called}


def print_agent_stats(result: dict, agent_name: str = "Agent", model_name: str = ""):
    """
    Stampa le statistiche di un agente in formato box.

    ╔══════════════════════════════════════════════════════════╗
    ║  AGENT: KnowledgeAgent  │  MODEL: qwen2.5:7b            ║
    ╠══════════════════════════════════════════════════════════╣
    ║  PROMPT  Who won the Tour de France 2025? [...]          ║
    ╠══════════════════════════════════════════════════════════╣
    ║  TOOLS (1)                                               ║
    ║  └─ duckduckgo_search                                    ║
    ║     query: Tour de France 2025 winner                    ║
    ╠══════════════════════════════════════════════════════════╣
    ║  TOKENS   in: 1163   out: 2   tot: 1165                  ║
    ║  TIME     prompt: 0.30s   gen: 0.02s   tot: 0.51s        ║
    ╠══════════════════════════════════════════════════════════╣
    ║  OUTPUT   B                                              ║
    ╚══════════════════════════════════════════════════════════╝
    """
    W = 120  # larghezza interna del box

    def row(text=""):
        return f"║  {text:<{W - 2}}║"

    def div():
        return "╠" + "═" * W + "╣"

    messages  = result.get("messages", [])
    ai_msg    = messages[-1] if messages else None
    human_msg = messages[0]  if messages else None
    meta      = getattr(ai_msg, "response_metadata", {}) if ai_msg else {}
    display_model = model_name or meta.get("model", "unknown")

    print()
    print("╔" + "═" * W + "╗")
    header = f"AGENT: {agent_name}  │  MODEL: {display_model}"
    print(row(header[: W - 2]))
    print(div())

    # Prompt preview
    prompt_text = human_msg.content.replace("\n", " ") if human_msg else ""
    limit = W - 12
    prompt_preview = prompt_text[:limit] + "..." if len(prompt_text) > limit else prompt_text
    print(row(f"PROMPT  {prompt_preview}"))
    print(div())

    # Tool calls
    tool_calls = [
        tc
        for msg in messages
        if hasattr(msg, "tool_calls") and msg.tool_calls
        for tc in msg.tool_calls
    ]
    print(row(f"TOOLS ({len(tool_calls)})"))
    if tool_calls:
        for tc in tool_calls:
            print(row(f"  └─ {tc.get('name', '?')}"))
            for k, v in tc.get("args", {}).items():
                v_str = str(v)
                v_preview = v_str[: W - 8] + "..." if len(v_str) > W - 8 else v_str
                print(row(f"     {k}: {v_preview}"))
    else:
        print(row("  (nessun tool — risposta diretta)"))
    print(div())

    # Tokens
    usage = getattr(ai_msg, "usage_metadata", {}) if ai_msg else {}
    if usage:
        tok = f"TOKENS   in: {usage.get('input_tokens', 0)}   out: {usage.get('output_tokens', 0)}   tot: {usage.get('total_tokens', 0)}"
        print(row(tok))

    # Timing
    if meta and "total_duration" in meta:
        tot_s    = meta["total_duration"]                  / 1e9
        prompt_s = meta.get("prompt_eval_duration", 0)    / 1e9
        eval_s   = meta.get("eval_duration", 0)           / 1e9
        print(row(f"TIME     prompt: {prompt_s:.2f}s   gen: {eval_s:.2f}s   tot: {tot_s:.2f}s"))

    print(div())
    output_text = (ai_msg.content.replace("\n", " ") if ai_msg else "")[:W - 9]
    print(row(f"OUTPUT   {output_text}"))
    print("╚" + "═" * W + "╝")

print("Utils loaded")

## 8 · Tool matematici (MathAgent)

In [ ]:
import math
import re as _re


@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression safely.

    Args:
        expression (str): A mathematical expression string.
            Supported: +, -, *, /, **, sqrt(), log(), log10(),
            sin(), cos(), tan(), pi, e, abs(), round(), pow(),
            factorial(), gcd(), comb(), perm().

    Returns:
        str: The numeric result as a string, or an error message
            starting with 'Error:' if the expression is invalid.

    Example:
        Input: "sqrt(144) + 3**2"
        Output: "21.0"
    """
    allowed = {
        "sqrt": math.sqrt,   "log": math.log,     "log10": math.log10,
        "sin":  math.sin,    "cos": math.cos,      "tan":  math.tan,
        "pi":   math.pi,     "e":   math.e,
        "abs":  abs,         "round": round,       "pow":  pow,
        "factorial": math.factorial, "gcd": math.gcd,
        "comb": math.comb,   "perm": math.perm,
    }
    sanitized = _re.sub(r"[^0-9\+\-\*\/\%\(\)\.\,\s\_a-zA-Z]", "", expression)
    print(f"TOOL[calculator] - evaluating: {sanitized}")
    try:
        result = eval(sanitized, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as ex:
        return f"Error: {ex}"


@tool
def modular_arithmetic(a: int, b: int, mod: int, operation: str = "multiply") -> str:
    """
    Perform modular arithmetic: (a OP b) mod n.

    Args:
        a (int): First operand.
        b (int): Second operand.
        mod (int): The modulus n.
        operation (str): One of 'add', 'subtract', 'multiply', 'power'.
            Defaults to 'multiply'.

    Returns:
        str: The result of (a OP b) mod n as a string, or an error message
            if the operation is unknown.

    Example:
        Input: a=12, b=16, mod=24, operation="multiply"
        Output: "0"
    """
    ops = {
        "add":      (a + b) % mod,
        "subtract": (a - b) % mod,
        "multiply": (a * b) % mod,
        "power":    pow(a, b, mod),
    }
    if operation not in ops:
        return f"Unknown operation '{operation}'. Use: add, subtract, multiply, power."
    print(f"TOOL[modular_arithmetic] - {a} {operation} {b} mod {mod} = {ops[operation]}")
    return str(ops[operation])


@tool
def statistics_helper(data: str, operation: str) -> str:
    """
    Compute basic statistics on a list of numbers.

    Args:
        data (str): Comma-separated numeric values.
        operation (str): One of 'mean', 'median', 'std', 'variance', 'range', 'sort'.

    Returns:
        str: The computed statistic as a string, or an error message
            if the data is malformed or the operation is unknown.

    Example:
        Input: data="86,94,97,88,96", operation="median"
        Output: "94"
    """
    import statistics
    try:
        nums = [float(x.strip()) for x in data.split(",")]
    except ValueError:
        return "Error: provide comma-separated numbers."
    ops = {
        "mean":     lambda d: statistics.mean(d),
        "median":   lambda d: statistics.median(d),
        "std":      lambda d: statistics.stdev(d) if len(d) > 1 else 0,
        "variance": lambda d: statistics.variance(d) if len(d) > 1 else 0,
        "range":    lambda d: max(d) - min(d),
        "sort":     lambda d: sorted(d),
    }
    if operation not in ops:
        return f"Unknown operation. Use: {list(ops.keys())}"
    res = ops[operation](nums)
    print(f"TOOL[statistics_helper] - {operation}({data}) = {res}")
    return str(res)


math_tools = [calculator, modular_arithmetic, statistics_helper]
print("Math tools loaded")

## 9 · Tool di ricerca (KnowledgeAgent)

In [ ]:
@tool
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia and return the summary of the most relevant article.
    Use for stable facts: history, science, geography, famous people, definitions.
    Prefer over duckduckgo_search when the topic is not a recent event.

    Args:
        query (str): The search query.

    Returns:
        str: The first 2000 characters of the article summary, or an error message
            if no page is found or a disambiguation error occurs.

    Example:
        Input: "French Revolution"
        Output: "The French Revolution was a period of political and societal change in France..."
    """
    import wikipedia
    try:
        wikipedia.set_lang("en")
        results = wikipedia.search(query, results=3)
        if not results:
            return "No Wikipedia results found."
        for title in results:
            try:
                page = wikipedia.page(title, auto_suggest=False)
                print(f"TOOL[wikipedia_search] - found: {page.title} — {page.summary[:50]}...")
                return page.summary[:2000]
            except wikipedia.DisambiguationError:
                continue
        return "Could not retrieve a Wikipedia page."
    except Exception as ex:
        return f"Wikipedia error: {ex}"


@tool
def duckduckgo_search(query: str) -> str:
    """
    Search the web via DuckDuckGo and return the top 3 result snippets.
    Use for recent events or facts post knowledge-cutoff.
    Prefer over wikipedia_search when the topic is a recent or current event.

    Args:
        query (str): The search query.

    Returns:
        str: Up to 3 snippets formatted as '[title]\nbody', separated by
            blank lines. Returns 'No results found.' if the search yields
            nothing, or 'Search error: ...' on failure.

    Example:
        Input: "XXL Malta festival 2026"
        Output: "[XXL Malta 2026 — Official Announcement]\nXXL Malta returns to Fort Saint Elmo..."
    """
    from ddgs import DDGS
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No results found."
        print(f"TOOL[duckduckgo_search] - {len(results)} results for '{query}' → {results[0]['title']}")
        return "\n\n".join(f"[{r['title']}]\n{r['body']}" for r in results)
    except Exception as ex:
        return f"Search error: {ex}"


knowledge_tools = [wikipedia_search, duckduckgo_search]
print("Knowledge tools loaded")

## 10 · System prompts

In [ ]:
MATH_TOOLS_DESC      = render_text_description(math_tools)
KNOWLEDGE_TOOLS_DESC = render_text_description(knowledge_tools)

# ── MathAgent ────────────────────────────────────────────────────────────────
MATH_SYSTEM_PROMPT = f"""
You are the Math Specialist Agent for a quiz game.
Your ONLY job is to solve mathematical problems accurately using tools.
YOU MUST NEVER calculate in your head — always delegate to a tool.

AVAILABLE TOOLS:
{MATH_TOOLS_DESC}

RULES:
1. For ANY calculation, call the appropriate tool. No mental math.
2. If the problem requires multiple steps, chain tool calls one by one.
3. Once you have the final result, match it against the answer choices.
4. Your final output must be EXACTLY one character: A, B, C, or D. No other text, no explanation.
"""

# ── KnowledgeAgent ───────────────────────────────────────────────────────────
KNOWLEDGE_SYSTEM_PROMPT = f"""
You are the Factual Knowledge Specialist Agent for a quiz game.
Your ONLY job is to verify facts using tools — history, science, geography, culture, current events.
YOU MUST NEVER answer from internal memory — always verify with a tool first.

AVAILABLE TOOLS:
{KNOWLEDGE_TOOLS_DESC}

RULES:
1. Always search for the core subject of the question before answering.
2. Prefer wikipedia_search for stable facts; use duckduckgo_search for recent events.
3. If Wikipedia returns no useful result, fall back to duckduckgo_search.
4. Once you have the result, match it against the answer choices.
5. Your final output must be EXACTLY one character: A, B, C, or D. No other text, no explanation.
"""

print("System prompts ready")

## 11 · Factory — sotto-agenti

`build_sub_agents(model_config)` crea MathAgent e KnowledgeAgent.
Ri-esegui questa cella (e la successiva) per cambiare modello senza riavviare il kernel.

In [ ]:
def build_sub_agents(model_config: dict) -> tuple:
    """
    Crea math_agent e knowledge_agent con i modelli specificati in model_config.

    Args:
        model_config (dict): es. {"math": "qwen2.5:7b", "knowledge": "qwen2.5:1.5b", ...}

    Returns:
        tuple: (math_agent, knowledge_agent) — CompiledStateGraph LangGraph
    """
    _math = create_agent(
        model=make_model(model_config["math"]),
        system_prompt=MATH_SYSTEM_PROMPT,
        tools=math_tools,
    )
    _knowledge = create_agent(
        model=make_model(model_config["knowledge"]),
        system_prompt=KNOWLEDGE_SYSTEM_PROMPT,
        tools=knowledge_tools,
    )
    print(f"  math_agent      → {model_config['math']}")
    print(f"  knowledge_agent → {model_config['knowledge']}")
    return _math, _knowledge


math_agent, knowledge_agent = build_sub_agents(MODEL_CONFIG)
print("Sub-agents created")

## 12 · Factory — orchestratore e wrapper tool

I tool `ask_math_agent` / `ask_knowledge_agent` wrappano i sotto-agenti
e popolano `_question_trace` con la traccia grezza di ogni agente invocato.

In [ ]:
# Raccoglie la traccia grezza di ogni sotto-agente per la domanda corrente.
# Viene azzerato in answer_quiz_question prima di ogni chiamata.
_question_trace: dict = {}


@tool
def ask_math_agent(question_with_choices: str) -> str:
    """
    Delegate a mathematical question to the Math specialist agent.

    Use for: arithmetic, algebra, calculus, statistics, probability,
    modular arithmetic, combinatorics, geometry — any question that
    requires computation or numerical reasoning.

    Args:
        question_with_choices (str): The full question text including all
            answer choices (A, B, C, D).

    Returns:
        str: A single letter: "A", "B", "C", or "D".

    Example:
        Input: "What is sqrt(144) + 3**2?\nA: 20\nB: 21\nC: 15\nD: 18"
        Output: "B"
    """
    result = math_agent.invoke(
        {"messages": [("human", question_with_choices)]},
        config={"recursion_limit": 12},
    )
    _question_trace["math"] = _extract_agent_trace(result, MODEL_CONFIG["math"])
    print_agent_stats(result, "MathAgent", MODEL_CONFIG["math"])
    return result["messages"][-1].content


@tool
def ask_knowledge_agent(question_with_choices: str) -> str:
    """
    Delegate a factual knowledge question to the Knowledge specialist agent.

    Use for: history, geography, science, literature, general culture,
    current events, famous people, definitions — any question that is
    NOT purely mathematical or computational.

    Args:
        question_with_choices (str): The full question text including all
            answer choices (A, B, C, D).

    Returns:
        str: A single letter: "A", "B", "C", or "D".

    Example:
        Input: "Who painted the Sistine Chapel?\nA: Raphael\nB: Michelangelo\nC: Da Vinci\nD: Botticelli"
        Output: "B"
    """
    result = knowledge_agent.invoke(
        {"messages": [("human", question_with_choices)]},
        config={"recursion_limit": 8},
    )
    _question_trace["knowledge"] = _extract_agent_trace(result, MODEL_CONFIG["knowledge"])
    print_agent_stats(result, "KnowledgeAgent", MODEL_CONFIG["knowledge"])
    return result["messages"][-1].content


# ── Orchestratore ─────────────────────────────────────────────────────────────
orchestrator_tools      = [ask_math_agent, ask_knowledge_agent]
ORCHESTRATOR_TOOLS_DESC = render_text_description(orchestrator_tools)

ORCHESTRATOR_SYSTEM_PROMPT = f"""
You are the strict Routing Coordinator for a Quiz Game.
Your ONLY purpose is to route the question to the correct specialist and relay their answer.

Available Tools:
{ORCHESTRATOR_TOOLS_DESC}

WORKFLOW — follow these two steps in order:

STEP 1 — ROUTING (your first action):
  - Call ask_math_agent for questions requiring calculation or numerical reasoning.
  - Call ask_knowledge_agent for all other questions (history, science, culture, events, etc.).
  - Pass the ENTIRE question text and all 4 options (A, B, C, D) to the tool.
  - NEVER write tool calls as JSON or plain text. ALWAYS use the native tool calling mechanism.

STEP 2 — ANSWER (after the tool returns):
  - Read the letter returned by the tool.
  - Output ONLY that single letter (A, B, C, or D).
  - Do NOT call another tool.
  - Do NOT add any explanation, tag, punctuation, or whitespace.
"""


def build_orchestrator(model_config: dict):
    """
    Crea l'orchestratore con il modello specificato in model_config["orchestrator"].
    Deve essere chiamato dopo aver definito ask_math_agent e ask_knowledge_agent.

    Args:
        model_config (dict): es. {"orchestrator": "qwen2.5:7b", ...}

    Returns:
        CompiledStateGraph LangGraph
    """
    _orch = create_agent(
        model=make_model(model_config["orchestrator"]),
        system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
        tools=orchestrator_tools,
    )
    print(f"  orchestrator    → {model_config['orchestrator']}")
    return _orch


orchestrator = build_orchestrator(MODEL_CONFIG)
print("Orchestrator ready")

## 13 · Wrapper quiz

In [ ]:
def format_question(question: str, choices: dict) -> str:
    """Formatta domanda + opzioni come stringa per l'orchestratore."""
    lines = [question, "Options:"]
    for letter, text in choices.items():
        lines.append(f"{letter}: {text}")
    return "\n".join(lines)


def extract_letter(raw: str) -> str:
    """
    Estrae A/B/C/D dalla risposta grezza.
    Fallback: lettera casuale tra A-D per evitare bias sistematico verso 'A'.
    """
    m = re.search(r"\b([A-Da-d])\b", raw.strip())
    return m.group(1).upper() if m else random.choice(["A", "B", "C", "D"])


def answer_quiz_question(question: str, choices: dict) -> dict:
    """
    Punto di ingresso per il quiz. Invoca l'orchestratore e raccoglie
    la traccia grezza di tutti gli agenti coinvolti.

    Parameters
    ----------
    question : str  — testo della domanda
    choices  : dict — {'A': '...', 'B': '...', 'C': '...', 'D': '...'}

    Returns
    -------
    dict con campi:
        answer_letter : str  — lettera scelta (A/B/C/D)
        answer_text   : str  — testo dell'opzione scelta
        raw           : str  — risposta grezza dell'orchestratore
        agent_trace   : dict — traccia per-agente con struttura:
            {
              "orchestrator": {"model": ..., "raw_output": ..., "tools_called": [...]},
              "math":         {"model": ..., "raw_output": ..., "tools_called": [...]},  # se invocato
              "knowledge":    {"model": ..., "raw_output": ..., "tools_called": [...]}   # se invocato
            }
    """
    global _question_trace
    _question_trace = {}

    prompt = format_question(question, choices)
    result = orchestrator.invoke(
        {"messages": [("human", prompt)]},
        config={"recursion_limit": 10},
    )
    _question_trace["orchestrator"] = _extract_agent_trace(result, MODEL_CONFIG["orchestrator"])
    print_agent_stats(result, "Orchestrator", MODEL_CONFIG["orchestrator"])

    raw    = result["messages"][-1].content
    letter = extract_letter(raw)
    return {
        "answer_letter": letter,
        "answer_text":   choices.get(letter, "?"),
        "raw":           raw,
        "agent_trace":   dict(_question_trace),
    }

## 14 · Test

Testa il sistema su domande note prima di avviare il game loop.

In [ ]:
# ── Test di conoscenza (DuckDuckGo) ──────────────────────────────────────────
q_knowledge = "Who won the general classification of the Tour de France in the year 2025?"
c_knowledge = {
    "A": "Jonas Vingegaard",
    "B": "Tadej Pogačar",
    "C": "Remco Evenepoel",
    "D": "Primož Roglič",
}
res = answer_quiz_question(q_knowledge, c_knowledge)
print(f"\nANSWER: {res['answer_letter']} → {c_knowledge.get(res['answer_letter'], '?')}")
print(f"Agent trace keys: {list(res['agent_trace'].keys())}")

In [ ]:
# ── Test matematico (calculator tool) ────────────────────────────────────────
q_math = "What is the value of sqrt(144) + 3 squared?"
c_math = {"A": "18", "B": "21", "C": "15", "D": "27"}
res_math = answer_quiz_question(q_math, c_math)
print(f"\nANSWER: {res_math['answer_letter']} → {c_math.get(res_math['answer_letter'], '?')}")
print(f"Agent trace keys: {list(res_math['agent_trace'].keys())}")

## 15 · Game loop

Esegue `NUM_RUNS` partite per ogni competizione e salva i risultati in `data/runs/`.

**Struttura del log per domanda** (compatibile con `analysis.ipynb`):
```json
{
  "level": 3,
  "prize": 500,
  "question_text": "...",
  "options": {"A": "...", "B": "...", "C": "...", "D": "..."},
  "llm_raw_output": "B",
  "agent_trace": {
    "orchestrator": {"model": "qwen2.5:7b", "raw_output": "B", "tools_called": [...]},
    "knowledge":    {"model": "qwen2.5:7b", "raw_output": "B", "tools_called": [...]}
  },
  "chosen_letter": "B",
  "correct": true,
  "timed_out": false,
  "time": 1.83
}
```

In [ ]:
# ── Crea la cartella del run e salva i metadati ───────────────────────────────
run_dir  = create_run_folder(MODEL_CONFIG["orchestrator"])
metadata = save_metadata(run_dir, MODEL_CONFIG, REASONING, RUN_COMMENT, TECHNIQUES, NUM_RUNS)

comps = client.competitions.list_all()
print("Competitions found:", len(comps))
all_comp_data = {}

LETTERS = ["A", "B", "C", "D"]

for comp in comps:
    print()
    print("=" * 60)
    print("Competition", comp.id, ":", comp.name)
    attempts_log = []

    for run_idx in range(1, NUM_RUNS + 1):
        print("  Run", run_idx, "/", NUM_RUNS, "...", end=" ", flush=True)
        session       = client.game.start(comp.id)
        questions_log = []
        max_time      = 0.0

        last_correct_level = 0
        total_earned       = 0

        while session.in_progress:
            question = session.current_question
            if question is None:
                break

            opts         = question.options[:4]
            options_dict = {LETTERS[i]: opts[i].text for i in range(len(opts))}

            # Livello e premio dalla GameState (question.level può essere 0)
            current_level = session.state.current_level
            current_prize = next(
                (ml.amount for ml in session.state.money_pyramid if ml.level == current_level),
                None,
            )

            t0     = time.time()
            result = answer_quiz_question(question.text, options_dict)
            chosen_idx = LETTERS.index(result["answer_letter"])
            chosen_opt = question.options[chosen_idx]
            answer     = session.answer(chosen_opt.id)
            elapsed    = time.time() - t0
            if elapsed > max_time:
                max_time = elapsed

            if answer.correct:
                last_correct_level = current_level
                total_earned      += current_prize

            questions_log.append({
                "level":          current_level,
                "prize":          current_prize,
                "question_text":  question.text,
                "options":        options_dict,
                "llm_raw_output": result["raw"],         # compatibilità celle analisi
                "agent_trace":    result["agent_trace"],  # traccia per-agente
                "chosen_letter":  result["answer_letter"],
                "correct":        answer.correct,
                "timed_out":      answer.timed_out,
                "time":           elapsed,
            })

        print(
            "earned=" + str(total_earned)
            + ", q=" + str(len(questions_log))
            + ", max_t=" + f"{max_time:.2f}s"
        )
        attempts_log.append({
            "attempt":     run_idx,
            "earned":      total_earned,
            "final_level": last_correct_level,
            "questions":   questions_log,
        })

    save_competition_result(run_dir, comp, attempts_log)
    all_comp_data[comp.id] = {
        "name":       comp.name,
        "max_levels": comp.max_levels,
        "attempts":   attempts_log,
    }

run_info = save_run_summary(run_dir, all_comp_data, metadata)
update_runs_log(RUNS_DIR, run_dir.name, run_info)
print()
print("Run complete! Overall accuracy:", f"{run_info['overall_accuracy']:.1%}")